# Project 94 — Local Notebook Refactor Assistant
## Analyze → Detect Smells → Suggest Refactors → Generate Clean Code

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Sample Messy Code

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

messy_code_samples = [
    {
        "name": "data_processor.py",
        "code": '''
import pandas as pd
def process(f):
    d = pd.read_csv(f)
    d = d.dropna()
    d = d[d['age'] > 0]
    d = d[d['age'] < 150]
    d['name'] = d['name'].str.strip().str.title()
    d['email'] = d['email'].str.lower()
    d['score'] = d['grade'] * 10 + d['bonus']
    d['level'] = d['score'].apply(lambda x: 'A' if x > 90 else ('B' if x > 80 else ('C' if x > 70 else 'F')))
    d.to_csv('output.csv')
    print('done')
    return d
'''
    },
    {
        "name": "api_handler.py",
        "code": '''
from flask import request, jsonify
def handle():
    try:
        data = request.json
        name = data['name']
        email = data['email']
        age = data['age']
        if not name or len(name) < 2:
            return jsonify({'error': 'bad name'}), 400
        if '@' not in email:
            return jsonify({'error': 'bad email'}), 400
        if age < 0 or age > 200:
            return jsonify({'error': 'bad age'}), 400
        # save to db
        import sqlite3
        conn = sqlite3.connect('users.db')
        conn.execute(f"INSERT INTO users VALUES ('{name}', '{email}', {age})")
        conn.commit()
        conn.close()
        return jsonify({'status': 'ok'})
    except Exception as e:
        return jsonify({'error': str(e)}), 500
'''
    },
    {
        "name": "report_gen.py",
        "code": '''
def report(data):
    r = ""
    r += "Report\n"
    r += "======\n"
    total = 0
    count = 0
    for item in data:
        total += item['amount']
        count += 1
        r += f"{item['name']}: ${item['amount']}\n"
    r += f"Total: ${total}\n"
    r += f"Average: ${total/count}\n"
    r += f"Items: {count}\n"
    print(r)
    return r
'''
    },
]
print(f"Code samples to refactor: {len(messy_code_samples)}")

## Step 2 — Code Smell Detection

In [ ]:
class CodeSmell(BaseModel):
    smell_type: str = Field(description="e.g., long_method, sql_injection, magic_number, poor_naming")
    severity: str = Field(description="critical, high, medium, low")
    location: str
    description: str
    fix_category: str = Field(description="security, readability, maintainability, performance")

class CodeAnalysis(BaseModel):
    file_name: str
    quality_score: float = Field(ge=0, le=10)
    smells: list[CodeSmell]
    positive_aspects: list[str]
    complexity_rating: str = Field(description="simple, moderate, complex")
    refactor_priority: str = Field(description="urgent, high, medium, low")

analyzer = llm.with_structured_output(CodeAnalysis)

analyses = []
for sample in messy_code_samples:
    analysis = analyzer.invoke(
        f"Analyze this code for smells and quality issues:\n\n"
        f"File: {sample['name']}\n```python\n{sample['code']}\n```"
    )
    analyses.append(analysis)
    print(f"\n{analysis.file_name}: score={analysis.quality_score}/10 priority={analysis.refactor_priority}")
    for smell in analysis.smells:
        icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🔵"}.get(smell.severity, "⚪")
        print(f"  {icon} {smell.smell_type}: {smell.description[:60]}")

## Step 3 — Generate Refactored Code

In [ ]:
refactor_prompt = ChatPromptTemplate.from_messages([
    ("system", "Refactor this code addressing ALL identified smells. "
     "Apply best practices: meaningful names, single responsibility, "
     "input validation, parameterized queries, type hints. "
     "Return ONLY the refactored Python code."),
    ("human", "Original code:\n```python\n{code}\n```\n\n"
     "Smells to fix:\n{smells}\n\nRefactored code:")
])
refactor_chain = refactor_prompt | llm | StrOutputParser()

for sample, analysis in zip(messy_code_samples, analyses):
    smells_desc = "\n".join(f"- [{s.severity}] {s.smell_type}: {s.description}" for s in analysis.smells)
    refactored = refactor_chain.invoke({
        "code": sample["code"],
        "smells": smells_desc,
    })
    print(f"\n{'='*50}")
    print(f"REFACTORED: {sample['name']}")
    print(f"{'='*50}")
    print(refactored[:500])

## Step 4 — Refactoring Report

In [ ]:
import pandas as pd

rows = []
for a in analyses:
    for smell in a.smells:
        rows.append({
            "file": a.file_name,
            "smell": smell.smell_type,
            "severity": smell.severity,
            "category": smell.fix_category,
            "priority": a.refactor_priority,
        })

df = pd.DataFrame(rows)
print("REFACTORING REPORT")
print("=" * 50)
print(f"Total smells: {len(df)}")
print(f"\nBy severity: {df['severity'].value_counts().to_dict()}")
print(f"By category: {df['category'].value_counts().to_dict()}")

print(f"\nFile quality scores:")
for a in analyses:
    bar = "█" * int(a.quality_score)
    print(f"  {a.file_name:<25} {bar} {a.quality_score}/10")

## What You Learned
- **Automated code smell detection** with severity levels
- **Quality scoring** for code files
- **AI-powered refactoring** with targeted fixes
- **Refactoring reports** for team prioritization